In [1]:
import pandas as pd
import numpy as np


from util import *
from Model import Sequence_Generator
from Preprocessing import food_dict, nutrient_data, incidence_data,candidate_data, tf_dataset, tf_dataset_update, kwargs

import tensorflow as tf
import copy
import csv
import time
import pickle

args : <function get_params at 0x000001E9DBF09310>


100%|██████████| 261/261 [00:01<00:00, 192.41it/s]


 
0 diets are deleted as they have more than 3 empty slots


100%|██████████| 1195/1195 [00:06<00:00, 193.44it/s]


 
0 diets are deleted as they have more than 3 empty slots


In [26]:
list(tf_dataset)

[<tf.Tensor: shape=(261, 19), dtype=int32, numpy=
 array([[3716, 3108, 2922, ..., 1046,  973, 3715],
        [3716, 3246, 2961, ..., 2068,  973, 3715],
        [3716, 3317, 2957, ..., 1111,  968, 3715],
        ...,
        [3716, 3307, 2937, ..., 1103,  973, 3715],
        [3716, 2914, 2934, ..., 2381,  984, 3715],
        [3716, 3321, 2939, ..., 2434,  973, 3715]])>]

In [5]:
# Parameter (2) Constant Parameter Initialization
num_epochs = kwargs['num_epochs']
lr = kwargs['lr']
batch_size = kwargs['batch_size']
target_buffer_size = kwargs['buffer_size']
target_buffer_update = kwargs['buffer_update']
buffer_change_num = kwargs['buffer_change_number']

buffer_idx = 0
per_epoch_rewards = []
target_buffer = [ [[] for i in range(target_buffer_size)] for i in range(len(list(tf_dataset))) ]

In [6]:
kwargs

{'fully-connected_layer': 'GRU',
 'attention': True,
 'embed_dim': 128,
 'fc_dim': 64,
 'learning': 'off-policy',
 'policy': 'target',
 'use_beta': True,
 'use_buffer': True,
 'buffer_change_number': 20,
 'buffer_size': 60,
 'buffer_update': 10,
 'num_epochs': 30000,
 'lr': 0.0005,
 'num_tokens': 3718,
 'batch_size': 261}

In [8]:
# Make the directory that contains the results of training.
createFolder('./results')

'''
Teacher-Forced REINFORCE (TFR)
'''
# # Generate inital states for input, hidden, and concat state in Encoder and Decoder.
# ## --- (1) Define Encoder that embeds food sequences
# encoder = Encoder(len(food_dict), BATCH_SIZE, **kwargs)
# init_input = np.zeros([BATCH_SIZE, 1])
# init_hidden = encoder.initialize_hidden_state()
# init_output, _ = encoder(init_input, init_hidden)

# ## --- (2) Define Decoder that predicts food sequences
# decoder = Decoder(len(food_dict), **kwargs)
# decoder(init_input, init_hidden, init_output)

## --- (3) Define save_dir which represents the directory where Checkpoint is stored. The directory name consists of the parameters that controls the training of model.
root_dir = './training_log/'
save_dir = createDir(root_dir, kwargs)

## --- (4) Check and save the parameters.
print(kwargs)
saveParams(save_dir + 'params', kwargs, food_dict, nutrient_data, incidence_data.numpy())
## --- (5) Define Checkpoint object
# Define checkpoint_prefix which is the prefix part of save_dir.
checkpoint_prefix = os.path.join(save_dir + '/checkpoints', "ckpt")

## --- (6) Define Generator object
diet_generator = Sequence_Generator(food_dict, nutrient_data, incidence_data, candidate_data, **kwargs)


# Define checkpoint object in the variable 'checkpoint'.
# checkpoint = tf.train.Checkpoint(encoder = encoder, decoder = decoder, kwargs = kwargs)
checkpoint = tf.train.Checkpoint(generator = diet_generator, params = kwargs)
start = time.time() # start time


You have ./results directory already
./training_log/with_attention/GRU/
You have ./training_log/with_attention/GRU/ directory already
You have ./training_log/with_attention/GRU//checkpoints directory already
You have ./training_log/with_attention/GRU//params directory already
{'fully-connected_layer': 'GRU', 'attention': True, 'embed_dim': 128, 'fc_dim': 64, 'learning': 'off-policy', 'policy': 'target', 'use_beta': True, 'use_buffer': True, 'buffer_change_number': 20, 'buffer_size': 60, 'buffer_update': 10, 'num_epochs': 30000, 'lr': 0.0005, 'num_tokens': 3718, 'batch_size': 261}


In [9]:
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings(action='ignore')

In [10]:
for epoch in tqdm(range(num_epochs)):

    # Initialize total cumulative loss
    full_batch_loss = 0

    # Initialize per batch mean reward.
    mean_rewards_per_batch = np.empty((0, 1))

    # Initialize the batch number.
    batch_num = 0

    # Do training
    for i in range(len(list(tf_dataset_update))):

        # Define i-th batch of original and (potential) update dataset.
        x = list(tf_dataset)[i]
        x_update = list(tf_dataset_update)[i]

        # Initialize (diet) sequences at every batch
        ## full_seq_len is the length of full sequence.
        full_seq_len = x.shape[1]
        real_seqs_all = np.empty((0, full_seq_len))
        pred_seqs_all = np.empty((0, full_seq_len))

        # Train Teacher-Forced REINFORCe (TFR)
        real_seqs, batch_loss, pred_seqs, _, _, synthetic_target, _ = diet_generator.train(x, x_update)
        real_seqs_all = np.append(real_seqs_all, real_seqs, axis = 0)  # stack real diet sequences from each batch
        pred_seqs_all = np.append(pred_seqs_all, pred_seqs, axis = 0)  # stack synthetic diet sequences generated based on each batch
        full_batch_loss += batch_loss

        # Fill it target_buffer with synthetic target at every batch and buffer according to batch_num and buffer_idx.
        target_buffer[batch_num][buffer_idx] = synthetic_target

        # Update batch_num
        batch_num += 1

        # (Full batch) Store nutrition scores and rewards of synthetic diets generated at each batch.
        scores = get_score_pandas(pred_seqs_all, food_dict, nutrient_data)
        rewards = np.apply_along_axis(get_reward_ver2, arr = scores, axis = 1, done = 0)[:, 0]

        # (Batch) Store nutrition scores and rewards of synthetic diets generated at each batch.
        # mean_rewards = np.mean(rewards)
        # mean_rewards_per_batch = np.append(mean_rewards_per_batch, mean_rewards)
    
    # Reset the value of buffer_idx as 0 when the buffer_idx gets eqaul to target_buffer_size.
    if (buffer_idx + 1) % target_buffer_size == 0:
        buffer_idx = 0

    # Move to the next buffer by increasing 'buffer_idx'.
    else:
        buffer_idx += 1

    # Update on-training dataset using target_buffer which is composed of synthetic diets.
    tf_dataset_update  = update_dataset(epoch, batch_size, target_buffer_update, target_buffer, tf_dataset_update, x, food_dict, nutrient_data)

    # Store the checkpoint.
    #if (epoch + 1) % 100 == 0:
    #    checkpoint.save(file_prefix = checkpoint_prefix)

    # (Full batch) Define per_epoch_rewards variable using rewards_matrix function.
    per_epoch_rewards = rewards_matrix(epoch, rewards)

    # (Batch) Define per_epoch_rewards variable using rewards_matrix function.
    # per_epoch_rewards = rewards_matrix(epoch, mean_rewards_per_batch)

    # Save the reward per epoch
    #reward_df = pd.DataFrame(per_epoch_rewards)
    #reward_df = reward_df.astype('float32')
    #reward_df.columns = ['epoch', 'reward', 'sample']
    #dir_file_name = save_reward_df(reward_df, "TFR", target_buffer_update, target_buffer_size, lr, num_epochs)

    # Compute loss per epoch
    epoch_loss = full_batch_loss / len(list(tf_dataset))

    if (epoch + 1) % 100 == 0:
        print('epoch : {}, epoch_loss : {}'.format(epoch, epoch_loss)) 
        print(' ')

        # 매 에포크 별 SL & RL 생성 결과 확인
        print('REAL 시퀀스 :', sequence_to_sentence(real_seqs_all, food_dict)[0])
        print('REAL 시퀀스의 영양수준:', get_reward_ver2(get_score_vector(real_seqs_all[0], nutrient_data), done = 0)[0])
        print(' ')
        print('생성 시퀀스 :', sequence_to_sentence(pred_seqs_all, food_dict)[0])
        print('생성 시퀀스의 영양수준:', get_reward_ver2(get_score_vector(pred_seqs_all[0], nutrient_data), done = 0)[0])
        print(' ')

        # Calculate the (nutrient) score of the real and generated diets.
        nutrient_real = np.apply_along_axis(get_score_vector, axis = 1, arr = np.array(real_seqs_all), nutrient_data = nutrient_data)
        nutrient_gen = np.apply_along_axis(get_score_vector, axis = 1, arr = pred_seqs_all, nutrient_data = nutrient_data)

        # Get reward-related information of true and generated diet sequences.
        reward_info_real = np.apply_along_axis(get_reward_ver2, axis = 1, arr = nutrient_real, done = 0)
        reward_info_gen = np.apply_along_axis(get_reward_ver2, axis = 1, arr = nutrient_gen, done = 0)

        # Calculate mean rewards of true and generated diet sequences.
        mean_true_reward = np.mean(reward_info_real[:, 0])
        mean_gen_reward = np.mean(reward_info_gen[:, 0])
        print('REAL 평균 보상 :', mean_true_reward)
        print('생성 평균 보상 :', mean_gen_reward)

        print(' ')
        print("total time :", time.time() - start)  # 현재시각 - 시작시간 = 실행 시간

  0%|          | 99/30000 [03:35<10:46:10,  1.30s/it]

epoch : 99, epoch_loss : 5.912436167399089
 
REAL 시퀀스 : ['시작', 'B소고기미역죽', 'B고구마쉐이크', 'B사과(75g)', 'S오이스틱', 'S복숭아(천도)-80g', '율무밥(55)', '애호박맑은국', '북어채소찜', '건새우애호박볶음', '비트초절이', 'S크로와상', 'S보리차', '율무밥(55)', '맑은바지락탕', '소고기양배추조림', '깻잎나물', '배추김치', '종료']
REAL 시퀀스의 영양수준: 11
 
생성 시퀀스 : ['시작', '종료', '종료', '호박감자수제비국', 'S(우유제외)토마토스파게티', '새우살미역국', 'S멜론(100g)', '매쉬드포테이토', 'B치즈스크램블에그', 'B복숭아호두스무디', 'S복숭아(백도)-60g', 'S복숭아(60g)', 'S보리차', '종료', '찹쌀현미밥(55)', 'S증편(40g)', 'S멜론(50g)', '옥수수채소튀김', '탕평채무침']
생성 시퀀스의 영양수준: 12
 


  0%|          | 100/30000 [03:47<37:39:49,  4.53s/it]

REAL 평균 보상 : 13.731800766283525
생성 평균 보상 : 9.64367816091954
 
total time : 230.87926506996155


  1%|          | 199/30000 [07:17<10:11:49,  1.23s/it]

...

REAL 평균 보상 : 13.816091954022989
생성 평균 보상 : 13.670498084291188
 
total time : 72048.75891804695


100%|█████████▉| 29999/30000 [20:04:20<00:01,  1.34s/it]

epoch : 29999, epoch_loss : 0.10052896870507134
 
REAL 시퀀스 : ['시작', 'B소고기미역죽', 'B고구마쉐이크', 'B사과(75g)', 'S오이스틱', 'S복숭아(천도)-80g', '율무밥(55)', '애호박맑은국', '북어채소찜', '건새우애호박볶음', '비트초절이', 'S크로와상', 'S보리차', '율무밥(55)', '맑은바지락탕', '소고기양배추조림', '깻잎나물', '배추김치', '종료']
REAL 시퀀스의 영양수준: 11
 
생성 시퀀스 : ['시작', 'B소고기미역죽', 'B고구마쉐이크', 'B사과(75g)', 'S오이스틱', 'S복숭아(천도)-80g', '율무밥(55)', '애호박맑은국', '북어채소찜', '건새우애호박볶음', '비트초절이', 'S크로와상', 'S보리차', '율무밥(55)', '맑은바지락탕', '소고기양배추조림', '깻잎나물', '배추김치', '종료']
생성 시퀀스의 영양수준: 11
 


100%|██████████| 30000/30000 [20:04:32<00:00,  2.41s/it]

REAL 평균 보상 : 13.78544061302682
생성 평균 보상 : 13.651340996168582
 
total time : 72275.61706662178


In [11]:
generated_diets = sequence_to_sentence(pred_seqs_all, food_dict)


In [12]:
#diet_data = pd.read_csv('./data/diet_data/diet68_boy_morning.csv', encoding = 'cp949')
new_df = pd.DataFrame(0, columns = range(0,19), index = range(len(generated_diets)))
for i in tqdm(range(len(generated_diets))):
    new_df.iloc[i] = generated_diets[i]

new_df

100%|██████████| 261/261 [00:00<00:00, 1789.06it/s]


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,시작,B소고기미역죽,B고구마쉐이크,B사과(75g),S오이스틱,S복숭아(천도)-80g,율무밥(55),애호박맑은국,북어채소찜,건새우애호박볶음,비트초절이,S크로와상,S보리차,율무밥(55),맑은바지락탕,소고기양배추조림,깻잎나물,배추김치,종료
1,시작,B흰죽,B두유(100ml),B과일샐러드,S자두(50g),S사과(35g),찰현미밥(55),얼갈이맑은국,소고기파채볶음,브로콜리초무침,단무지무침,S개피떡(40g),S(우유제외)수제블루베리우유,해물굴소스볶음밥,안매운닭개장,고등어구이,(우유제외)단호박두유찜,배추김치,종료
2,시작,B찹쌀도너츠,B검은콩두유(100ml),B복숭아(60g),S바나나(100g),S복숭아(황도)-80g,해물버섯덮밥,동태살무국,오징어불고기,사과미나리무침,나박김치,S증편(40g),S오미자아이스크림,밤밥(55),다슬기된장국,가자미살구이,모듬과일샐러드,나박김치,종료
3,시작,B으깬견과류고구마샐러드(요거트),B(우유제외)수제바나나두유,B참외(50g),S복숭아(황도)-80g,S멜론(50g),현미밥(55),팽이장국,수제비샐러드,상추배생채,석박지,S쌀과자,S오미자아이스크림,현미밥(55),건새우시금치국,고구마연근맛탕,무사과무침,깍두기,종료
4,시작,B소고기타락죽,B(우유제외)수제바나나두유,B사과(75g),S참외(50g),S멜론(50g),옥수수밥(63),온청포묵국,오리훈제구이,땅콩우엉무침,배추김치,S단호박샐러드,S복숭아호두스무디,옥수수밥(63),쑥갓어묵탕,칠리새우,상추배생채,비트초절이,종료
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256,시작,B해물죽,B고구마쉐이크,B블루베리고구마샐러드,S사과(35g),S사과샐러드(마요네즈),잡곡밥(55),대구지리,새우살채소볶음,건취나물간장볶음,열무물김치,S멸치주먹밥,S매실쥬스,잡곡밥(63),팽이장국,주꾸미채소간장볶음,진미채초무침,비트초절이,종료
257,시작,B시금치견과류주먹밥,B복숭아호두스무디,B복숭아(80g),S멜론(50g),S참외(50g),옥수수밥(63),쑥갓어묵국,수제비샐러드,애호박건새우볶음,배추김치,S시금치견과류주먹밥,S보리차,알리오올리오스파게티,애호박두부젓국찌개,토마토치즈범벅,진미채초무침,배추김치,종료
258,시작,B약과,B수제블루베리우유,B과일샐러드(요거트드레싱),S사과(35g),S블루베리고구마샐러드,검정콩밥(55),얼갈이배추국,매콤낙지채소볶음,건취나물간장볶음,오이소박이김치,S(우유제외)토마토스파게티,S보리차,검정콩밥(55),유부쑥갓맑은국,삼치구이+양념장,매쉬드포테이토,배추김치,종료
259,시작,B으깬견과류고구마샐러드(요거트),B(우유제외)수제바나나두유,B블루베리고구마샐러드,S복숭아(60g),S사과(100g),현미밥(55),동태살무국,닭가슴살고구마볶음,머위나물,오이소박이김치,S(우유제외)토마토스파게티,S코코아,현미밥(55),오징어숙주국,소고기파인애플볶음,건취나물간장볶음,파김치,종료


In [13]:
nutrient_reward = reward_info_gen[:, 0]
new_df['nutrient_score'] = nutrient_reward
new_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,nutrient_score
0,시작,B소고기미역죽,B고구마쉐이크,B사과(75g),S오이스틱,S복숭아(천도)-80g,율무밥(55),애호박맑은국,북어채소찜,건새우애호박볶음,비트초절이,S크로와상,S보리차,율무밥(55),맑은바지락탕,소고기양배추조림,깻잎나물,배추김치,종료,11
1,시작,B흰죽,B두유(100ml),B과일샐러드,S자두(50g),S사과(35g),찰현미밥(55),얼갈이맑은국,소고기파채볶음,브로콜리초무침,단무지무침,S개피떡(40g),S(우유제외)수제블루베리우유,해물굴소스볶음밥,안매운닭개장,고등어구이,(우유제외)단호박두유찜,배추김치,종료,15
2,시작,B찹쌀도너츠,B검은콩두유(100ml),B복숭아(60g),S바나나(100g),S복숭아(황도)-80g,해물버섯덮밥,동태살무국,오징어불고기,사과미나리무침,나박김치,S증편(40g),S오미자아이스크림,밤밥(55),다슬기된장국,가자미살구이,모듬과일샐러드,나박김치,종료,15
3,시작,B으깬견과류고구마샐러드(요거트),B(우유제외)수제바나나두유,B참외(50g),S복숭아(황도)-80g,S멜론(50g),현미밥(55),팽이장국,수제비샐러드,상추배생채,석박지,S쌀과자,S오미자아이스크림,현미밥(55),건새우시금치국,고구마연근맛탕,무사과무침,깍두기,종료,15
4,시작,B소고기타락죽,B(우유제외)수제바나나두유,B사과(75g),S참외(50g),S멜론(50g),옥수수밥(63),온청포묵국,오리훈제구이,땅콩우엉무침,배추김치,S단호박샐러드,S복숭아호두스무디,옥수수밥(63),쑥갓어묵탕,칠리새우,상추배생채,비트초절이,종료,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256,시작,B해물죽,B고구마쉐이크,B블루베리고구마샐러드,S사과(35g),S사과샐러드(마요네즈),잡곡밥(55),대구지리,새우살채소볶음,건취나물간장볶음,열무물김치,S멸치주먹밥,S매실쥬스,잡곡밥(63),팽이장국,주꾸미채소간장볶음,진미채초무침,비트초절이,종료,15
257,시작,B시금치견과류주먹밥,B복숭아호두스무디,B복숭아(80g),S멜론(50g),S참외(50g),옥수수밥(63),쑥갓어묵국,수제비샐러드,애호박건새우볶음,배추김치,S시금치견과류주먹밥,S보리차,알리오올리오스파게티,애호박두부젓국찌개,토마토치즈범벅,진미채초무침,배추김치,종료,14
258,시작,B약과,B수제블루베리우유,B과일샐러드(요거트드레싱),S사과(35g),S블루베리고구마샐러드,검정콩밥(55),얼갈이배추국,매콤낙지채소볶음,건취나물간장볶음,오이소박이김치,S(우유제외)토마토스파게티,S보리차,검정콩밥(55),유부쑥갓맑은국,삼치구이+양념장,매쉬드포테이토,배추김치,종료,14
259,시작,B으깬견과류고구마샐러드(요거트),B(우유제외)수제바나나두유,B블루베리고구마샐러드,S복숭아(60g),S사과(100g),현미밥(55),동태살무국,닭가슴살고구마볶음,머위나물,오이소박이김치,S(우유제외)토마토스파게티,S코코아,현미밥(55),오징어숙주국,소고기파인애플볶음,건취나물간장볶음,파김치,종료,14


In [15]:
real_df = pd.DataFrame(0, columns =range(0,19), index = range(len(generated_diets)))

for i in tqdm(range(len(generated_diets))):
    real_df.iloc[i] = sequence_to_sentence(real_seqs_all, food_dict)[i]

#real_df

100%|██████████| 261/261 [00:00<00:00, 531.20it/s]


In [16]:
real_reward = reward_info_real[:, 0]
real_df['nutrient_score'] = real_reward
real_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,nutrient_score
0,시작,B소고기미역죽,B고구마쉐이크,B사과(75g),S오이스틱,S복숭아(천도)-80g,율무밥(55),애호박맑은국,북어채소찜,건새우애호박볶음,비트초절이,S크로와상,S보리차,율무밥(55),맑은바지락탕,소고기양배추조림,깻잎나물,배추김치,종료,11
1,시작,B가지죽,B두유(100ml),B복숭아(60g),S자두(50g),S사과(35g),찰현미밥(55),얼갈이맑은국,소고기파채볶음,브로콜리초무침,단무지무침,S개피떡(40g),S(우유제외)수제블루베리우유,해물굴소스볶음밥,안매운닭개장,고등어구이,(우유제외)단호박두유찜,배추김치,종료,13
2,시작,B찹쌀도너츠,B검은콩두유(100ml),B복숭아(60g),S바나나(100g),S복숭아(황도)-80g,해물버섯덮밥,다진소고기콩나물국,오징어간장조림,사과미나리무침,나박김치,S증편(40g),S오미자아이스크림,밤밥(55),다슬기된장국,삼치살두부감자조림,모듬과일샐러드,나박김치,종료,14
3,시작,B으깬견과류고구마샐러드(요거트),B(우유제외)수제바나나두유,B참외(50g),S복숭아(황도)-80g,S멜론(50g),현미밥(55),팽이장국,수제비샐러드,상추배생채,석박지,S쌀과자,S오미자아이스크림,현미밥(55),건새우시금치국,고구마연근맛탕,무사과무침,깍두기,종료,15
4,시작,B채소전,B(우유제외)수제바나나두유,B사과(75g),S참외(50g),S멜론(50g),옥수수밥(63),온청포묵국,오리훈제구이,땅콩우엉무침,배추김치,S단호박샐러드,S복숭아호두스무디,옥수수밥(63),쑥갓어묵탕,칠리새우,상추배생채,비트초절이,종료,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256,시작,B해물죽,B고구마쉐이크,B블루베리고구마샐러드,S사과(35g),S사과샐러드(마요네즈),잡곡밥(55),대구지리,새우살채소볶음,건취나물간장볶음,열무물김치,S멸치주먹밥,S매실쥬스,잡곡밥(63),팽이장국,주꾸미채소간장볶음,진미채초무침,비트초절이,종료,15
257,시작,B시금치견과류주먹밥,B복숭아호두스무디,B복숭아(80g),S멜론(50g),S참외(50g),옥수수밥(63),동태살무국,수제비샐러드,애호박건새우볶음,배추김치,S시금치견과류주먹밥,S보리차,알리오올리오스파게티,애호박두부젓국찌개,토마토치즈범벅,진미채초무침,배추김치,종료,14
258,시작,B약과,B수제블루베리우유,B과일샐러드(요거트드레싱),S사과(35g),S블루베리고구마샐러드,검정콩밥(55),얼갈이배추국,매콤낙지채소볶음,건취나물간장볶음,오이소박이김치,S(우유제외)토마토스파게티,S보리차,검정콩밥(55),유부쑥갓맑은국,삼치구이+양념장,매쉬드포테이토,배추김치,종료,14
259,시작,B으깬견과류고구마샐러드(요거트),B(우유제외)수제바나나두유,B블루베리고구마샐러드,S복숭아(60g),S사과(100g),현미밥(55),동태살무국,닭가슴살고구마볶음,머위나물,오이소박이김치,S(우유제외)토마토스파게티,S코코아,현미밥(55),오징어숙주국,소고기파인애플볶음,건취나물간장볶음,파김치,종료,14


In [23]:
np.mean(new_df.nutrient_score), np.mean(real_df.nutrient_score)

(13.651340996168582, 13.78544061302682)

In [19]:
variance_score = 0


new = new_df[new_df.columns[1:18]]
real = real_df[real_df.columns[1:18]]

for i in range(len(new)):
    for j in new.columns:
        
        if new.iloc[i][j] != real.iloc[i][j]:
            variance_score += 1
            
            
variance_score = variance_score/(len(new)*len(new.columns))

In [20]:
variance_score

0.028397565922920892

In [21]:
new_df.to_excel('6~8세, 남, 간편식_buffer60_change20_epoch=30000_lr=0.0005_beta14_syn15.xlsx')

In [22]:
real_df.to_excel('6~8세, 남, 간편식_buffer60_change20_epoch=30000_lr=0.0005_beta14_syn15_target.xlsx')